# BYOL on CIFAR-10

这个 Notebook 展示 `BYOL` 在 `CIFAR-10` 上的一个教学版实现，重点解释：

- online network 和 target network 的结构
- momentum update 为什么重要
- predictor 的作用
- BYOL loss 如何计算
- 为什么 BYOL 不需要显式负样本

## 1. 环境准备

如果本地环境尚未安装依赖，可以先执行：

```bash
pip install torch torchvision matplotlib
```

In [ ]:
# dataclass 用于集中管理实验配置
from dataclasses import dataclass

# matplotlib 用于样本和训练曲线可视化
import matplotlib.pyplot as plt
# PyTorch 核心模块
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
# torchvision 提供数据集、增强和 backbone
from torchvision import datasets, models, transforms

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)

# 优先使用 GPU，没有则退回 CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    data_root: str = './data'
    image_size: int = 224
    batch_size: int = 64
    num_workers: int = 2
    lr: float = 1e-3
    epochs: int = 5
    projection_dim: int = 256
    hidden_dim: int = 512
    momentum: float = 0.996
    linear_batch_size: int = 128
    linear_epochs: int = 3


cfg = Config()
cfg

## 2. 双视图数据增强

BYOL 也会对同一张图生成两个视图，但和 SimCLR / MoCo 不同，BYOL 没有显式使用负样本。

In [ ]:
# CIFAR-10 常用归一化参数
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2470, 0.2435, 0.2616)

# BYOL 也依赖较强数据增强来制造两个不同视图
contrastive_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.RandomResizedCrop(cfg.image_size, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)
    ], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

eval_train_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

eval_test_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

In [ ]:
class PairDataset(Dataset):
    def __init__(self, root, train=True, transform=None, download=True):
        self.dataset = datasets.CIFAR10(root=root, train=train, download=download)
        self.transform = transform
        self.classes = self.dataset.classes

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        view1 = self.transform(image)
        view2 = self.transform(image)
        return view1, view2, label


contrastive_dataset = PairDataset(cfg.data_root, train=True, transform=contrastive_transform, download=True)
linear_train_dataset = datasets.CIFAR10(cfg.data_root, train=True, transform=eval_train_transform, download=True)
linear_test_dataset = datasets.CIFAR10(cfg.data_root, train=False, transform=eval_test_transform, download=True)
classes = contrastive_dataset.classes
classes

In [ ]:
contrastive_loader = DataLoader(
    contrastive_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)

linear_train_loader = DataLoader(
    linear_train_dataset,
    batch_size=cfg.linear_batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

linear_test_loader = DataLoader(
    linear_test_dataset,
    batch_size=cfg.linear_batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

## 3. BYOL 模型结构

BYOL 的核心是两套网络：

1. `online network`
   - 包含 encoder、projector、predictor。

2. `target network`
   - 包含 encoder、projector。
   - 不直接做反向传播，而是通过动量方式从 online network 更新。

和 MoCo 不同，BYOL 没有 queue，也没有显式负样本。

In [ ]:
class MLPHead(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)


class EncoderProjector(nn.Module):
    def __init__(self, projection_dim=256, hidden_dim=512):
        super().__init__()
        backbone = models.resnet18(weights=None)
        in_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.projector = MLPHead(in_dim, hidden_dim, projection_dim)
        self.feature_dim = in_dim

    def forward(self, x):
        h = self.backbone(x)
        z = self.projector(h)
        return h, z


class BYOL(nn.Module):
    def __init__(self, projection_dim=256, hidden_dim=512, momentum=0.996):
        super().__init__()
        self.online_network = EncoderProjector(projection_dim=projection_dim, hidden_dim=hidden_dim)
        self.target_network = EncoderProjector(projection_dim=projection_dim, hidden_dim=hidden_dim)
        self.predictor = MLPHead(projection_dim, hidden_dim, projection_dim)
        self.momentum = momentum
        self.feature_dim = self.online_network.feature_dim

        # 初始化时让 target network 与 online network 参数一致
        for param_o, param_t in zip(self.online_network.parameters(), self.target_network.parameters()):
            param_t.data.copy_(param_o.data)
            param_t.requires_grad = False

    @torch.no_grad()
    def momentum_update_target_network(self):
        # target network 通过动量方式平滑跟随 online network
        for param_o, param_t in zip(self.online_network.parameters(), self.target_network.parameters()):
            param_t.data = param_t.data * self.momentum + param_o.data * (1.0 - self.momentum)

    def forward(self, view1, view2):
        # online 分支
        h1, z1 = self.online_network(view1)
        h2, z2 = self.online_network(view2)
        p1 = self.predictor(z1)
        p2 = self.predictor(z2)

        with torch.no_grad():
            # target 分支只提供目标表示，不参与梯度回传
            t_h1, t_z1 = self.target_network(view1)
            t_h2, t_z2 = self.target_network(view2)

        return h1, h2, p1, p2, t_z1, t_z2


model = BYOL(
    projection_dim=cfg.projection_dim,
    hidden_dim=cfg.hidden_dim,
    momentum=cfg.momentum,
).to(device)
model

## 4. BYOL loss 怎么计算

BYOL 不用负样本，而是让 online 分支去预测 target 分支的表示。

常见写法是对归一化后的向量做均方差等价形式：

$$
\mathcal{L}(p, z) = 2 - 2 \cdot \frac{p}{\|p\|_2} \cdot \frac{z}{\|z\|_2}
$$

其中：
- `p` 是 online network 经 predictor 后的输出
- `z` 是 target network 的输出

完整损失一般是双向平均：

$$
\mathcal{L}_{BYOL} = \frac{1}{2} \mathcal{L}(p_1, z_2) + \frac{1}{2} \mathcal{L}(p_2, z_1)
$$

含义是：
- 用 view1 的 online 输出去预测 view2 的 target 输出
- 再反过来做一次

虽然没有显式负样本，但 online / target 非对称结构、predictor 和动量更新一起帮助模型避免简单塌缩。

In [ ]:
def byol_loss_fn(p, z):
    # 对两个向量做 L2 归一化，后面可以直接比较方向一致性
    p = F.normalize(p, dim=1)
    z = F.normalize(z, dim=1)
    # 这里的写法等价于 2 - 2 * cosine_similarity 的 batch 平均
    return 2 - 2 * (p * z.detach()).sum(dim=1).mean()


view1, view2, _ = next(iter(contrastive_loader))
view1 = view1.to(device)
view2 = view2.to(device)

with torch.no_grad():
    _, _, p1, p2, t_z1, t_z2 = model(view1, view2)
    sample_loss = 0.5 * byol_loss_fn(p1, t_z2) + 0.5 * byol_loss_fn(p2, t_z1)

print('p1 shape:', p1.shape)
print('p2 shape:', p2.shape)
print('t_z1 shape:', t_z1.shape)
print('t_z2 shape:', t_z2.shape)
print('sample loss:', float(sample_loss))

## 5. 训练函数

In [ ]:
optimizer = optim.Adam(list(model.online_network.parameters()) + list(model.predictor.parameters()), lr=cfg.lr)


def train_one_epoch_byol(model, dataloader, optimizer, device):
    model.train()
    running_loss = 0.0
    total = 0

    for view1, view2, _ in dataloader:
        view1 = view1.to(device)
        view2 = view2.to(device)

        optimizer.zero_grad()
        _, _, p1, p2, t_z1, t_z2 = model(view1, view2)

        # 双向对齐：online(view1) 预测 target(view2)，online(view2) 预测 target(view1)
        loss = 0.5 * byol_loss_fn(p1, t_z2) + 0.5 * byol_loss_fn(p2, t_z1)
        loss.backward()
        optimizer.step()

        # 每轮参数更新后，再把 target network 朝 online network 做一次动量更新
        model.momentum_update_target_network()

        batch_size = view1.size(0)
        running_loss += loss.item() * batch_size
        total += batch_size

    return running_loss / total

In [ ]:
history = []

for epoch in range(cfg.epochs):
    epoch_loss = train_one_epoch_byol(model, contrastive_loader, optimizer, device)
    history.append(epoch_loss)
    print(f'Epoch [{epoch + 1}/{cfg.epochs}] byol_loss={epoch_loss:.4f}')

In [ ]:
epochs = range(1, len(history) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, history, marker='o')
plt.title('BYOL loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

## 6. 提取 encoder 特征并做线性评估

BYOL 预训练后，同样更关心 online encoder 学到的表征质量。

In [ ]:
@torch.no_grad()
def extract_features(backbone, dataloader, device):
    backbone.eval()
    features = []
    labels = []
    for images, target in dataloader:
        images = images.to(device)
        feats = backbone(images)
        features.append(feats.cpu())
        labels.append(target)
    return torch.cat(features, dim=0), torch.cat(labels, dim=0)


train_features, train_labels = extract_features(model.online_network.backbone, linear_train_loader, device)
test_features, test_labels = extract_features(model.online_network.backbone, linear_test_loader, device)

print('train_features:', train_features.shape)
print('test_features:', test_features.shape)

In [ ]:
class FeatureDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features.float()
        self.labels = labels.long()

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


train_feature_loader = DataLoader(FeatureDataset(train_features, train_labels), batch_size=cfg.linear_batch_size, shuffle=True)
test_feature_loader = DataLoader(FeatureDataset(test_features, test_labels), batch_size=cfg.linear_batch_size, shuffle=False)

linear_head = nn.Linear(model.feature_dim, 10).to(device)
linear_optimizer = optim.Adam(linear_head.parameters(), lr=1e-3)
linear_criterion = nn.CrossEntropyLoss()

In [ ]:
def train_one_epoch_linear(head, dataloader, criterion, optimizer, device):
    head.train()
    running_loss = 0.0
    running_correct = 0
    total = 0
    for features, labels in dataloader:
        features = features.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = head(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * features.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, running_correct / total


@torch.no_grad()
def evaluate_linear(head, dataloader, criterion, device):
    head.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0
    for features, labels in dataloader:
        features = features.to(device)
        labels = labels.to(device)
        logits = head(features)
        loss = criterion(logits, labels)
        running_loss += loss.item() * features.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, running_correct / total

In [ ]:
for epoch in range(cfg.linear_epochs):
    train_loss, train_acc = train_one_epoch_linear(linear_head, train_feature_loader, linear_criterion, linear_optimizer, device)
    test_loss, test_acc = evaluate_linear(linear_head, test_feature_loader, linear_criterion, device)
    print(
        f'Epoch [{epoch + 1}/{cfg.linear_epochs}] '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} '
        f'test_loss={test_loss:.4f} test_acc={test_acc:.4f}'
    )

## 7. BYOL 为什么没有负样本也能工作

从实现角度看，BYOL 不是简单地让两个分支互相复制，而是引入了几层非对称设计：

- online / target 双网络不完全对称
- online 多了一个 predictor
- target 通过动量更新，不直接反向传播

这些设计共同减少了“所有输出都塌缩成同一个常数向量”的风险。

和 SimCLR / MoCo 相比：
- BYOL 不显式构造负样本
- 训练目标更像“预测另一个视图的目标表示”
- 工程上不需要大 queue，也不需要大 batch 的负样本池